# iAuditAge - SALD held-out test (self-checking)

**Why SALD:** the BrainAgeNeXt paper used SALD only as a *test* set, so unlike IXI and eNKI it is
not in BrainAgeNeXt's training data. This notebook measures both CNNs on 200 SALD scans
(age-balanced, ages 20-80).

**How to run:** *Runtime > Change runtime type >* **T4 GPU**, then *Runtime > Run all*, allow Drive.
If Colab disconnects, *Run all* again - every finished scan is saved to
`MyDrive/iAuditAge/sald_outputs/` and never redone. Expected: ~150 s/scan, about 8 h.

**Nothing is tuned on SALD:**
- BrainAgeNeXt orientation stays locked to the IXI run's choice (**as_is**).
- Ridge heads are trained only on the IXI Guys+HH features from the earlier run
  (`MyDrive/iAuditAge/outputs/`) and applied to SALD unchanged.

**Safety checks** (the run stops itself, logs why, and frees the runtime):
- every scan is verified against S3's checksum (multipart ETag); IXI002 against its MD5
- pre-flight: IXI002 is re-run each session and must reproduce the laptop (DeepBrainNet +/-1 y,
  features r>=0.98; BrainAgeNeXt +/-1.5 y in all 4 orientations)
- watchdogs: 3 failures in a row or >5% failed; prediction collapse (near-constant output);
  implausible running error (DeepBrainNet > 20 y, BrainAgeNeXt > 15 y over 20 scans)
- every saved file is re-loaded and shape/NaN-checked before it counts as done

When it finishes, run only the last cell (export) and send the zip.

In [ ]:
# ===== Settings (defaults are the safe choice; normally change nothing) =====
AUTO_RELEASE_RUNTIME = True   # free the Colab machine when finished or when a safety check stops the run
DRIVE_FOLDER = 'iAuditAge'

In [ ]:
# ===== 1. Install (pinned to the versions validated on the laptop) =====
import sys, os, subprocess, importlib.metadata as im

class Stop(Exception):
    pass

WATCH = ('numpy', 'scipy', 'pandas')
loaded = [p for p in WATCH if p in sys.modules]
before = {p: im.version(p) for p in WATCH}
tfv = '.'.join(im.version('tensorflow').split('.')[:2])
pkgs = ['antspyx==0.6.3', 'antspynet==0.3.2', f'tf_keras~={tfv}.0', 'monai==1.6.1', 'torchio==1.2.1',
        'git+https://github.com/FrancescoLR/MedNeXt.git@463813b1c7352edb487f43b1eecebe507429a195']
print('installing (about 2-4 min) ...', flush=True)
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], capture_output=True, text=True)
if r.returncode:
    print(r.stdout[-2000:], r.stderr[-4000:])
    raise Stop('pip install failed - nothing was run. Send this output.')
env = dict(os.environ, TF_USE_LEGACY_KERAS='1', TF_CPP_MIN_LOG_LEVEL='3')
chk = subprocess.run([sys.executable, '-c',
                      'import ants, antspynet, tf_keras, monai, torchio, nnunet_mednext; print("imports ok")'],
                     capture_output=True, text=True, env=env)
print(chk.stdout.strip() or chk.stderr[-3000:])
if chk.returncode:
    raise Stop('installed packages do not import - nothing was run. Send this output.')
changed = [p for p in loaded if im.version(p) != before[p]]
if changed:
    print(f'{changed} changed after already being loaded -> restarting the runtime once. Click Run all again.')
    os.kill(os.getpid(), 9)

In [ ]:
# ===== 2. Embedded data (do not edit) =====
SALD_CSV = "scan_id,dataset,site,chronological_age,sex,fname,url,etag,size\r\nSALD_031274,SALD,SALD,72.0,F,sub-031274_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031274/anat/sub-031274_T1w.nii.gz,d2fd6b05fcc794ff984c8f0273a5db5a-2,10417357\r\nSALD_031275,SALD,SALD,44.0,F,sub-031275_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031275/anat/sub-031275_T1w.nii.gz,6373644b0dc899747abce0eeeec40fb2-2,10618302\r\nSALD_031276,SALD,SALD,74.0,F,sub-031276_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031276/anat/sub-031276_T1w.nii.gz,b5cc1a0d6f53a9bd1b1cf913ff8f5a77-2,10834666\r\nSALD_031277,SALD,SALD,60.0,F,sub-031277_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031277/anat/sub-031277_T1w.nii.gz,cefb0f1ff13b66a0e42b16d414f6c0e1-2,10133241\r\nSALD_031278,SALD,SALD,55.0,F,sub-031278_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031278/anat/sub-031278_T1w.nii.gz,499bd4abbf21a4795b9bb9f29f7fd980-2,10846245\r\nSALD_031281,SALD,SALD,65.0,F,sub-031281_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031281/anat/sub-031281_T1w.nii.gz,bec7ab90f347f40ad5e459977c804285-2,11400894\r\nSALD_031282,SALD,SALD,61.0,F,sub-031282_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031282/anat/sub-031282_T1w.nii.gz,5f370af07f72d1ce62ae622d3afb0af5-2,9936315\r\nSALD_031284,SALD,SALD,73.0,F,sub-031284_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031284/anat/sub-031284_T1w.nii.gz,f6bad496de38035c623eebfd1b3c8bb0-2,10074804\r\nSALD_031285,SALD,SALD,54.0,M,sub-031285_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031285/anat/sub-031285_T1w.nii.gz,98681600f1ea06193f7b787ca0066e03-2,10161495\r\nSALD_031286,SALD,SALD,76.0,M,sub-031286_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031286/anat/sub-031286_T1w.nii.gz,0003770bc64867fb49c5f00c512aef41-2,11424735\r\nSALD_031287,SALD,SALD,72.0,F,sub-031287_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031287/anat/sub-031287_T1w.nii.gz,11d4cc952facc8dd542f9c517a68e112-2,10210175\r\nSALD_031293,SALD,SALD,45.0,F,sub-031293_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031293/anat/sub-031293_T1w.nii.gz,d6d9d4c5afbd037227faf3c833f32c5b-2,10269037\r\nSALD_031294,SALD,SALD,48.0,F,sub-031294_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031294/anat/sub-031294_T1w.nii.gz,d6739fbfe5626f1f4e1babfacc846651-2,9729150\r\nSALD_031296,SALD,SALD,45.0,F,sub-031296_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031296/anat/sub-031296_T1w.nii.gz,2dade4bf4a5a17a11e23ffeb245cb5bc-2,10903479\r\nSALD_031298,SALD,SALD,59.0,M,sub-031298_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031298/anat/sub-031298_T1w.nii.gz,a5eb7fb2b54260d786cda35d70b6b43b-2,11352162\r\nSALD_031300,SALD,SALD,61.0,F,sub-031300_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031300/anat/sub-031300_T1w.nii.gz,bdf2ad72fa79cb33d0eb5bdd2bb3b625-2,10757779\r\nSALD_031304,SALD,SALD,71.0,M,sub-031304_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031304/anat/sub-031304_T1w.nii.gz,bb2093d020eec2a03d159b01f9b236a0-2,10419153\r\nSALD_031307,SALD,SALD,56.0,F,sub-031307_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031307/anat/sub-031307_T1w.nii.gz,87de4426877342edc79756f9ea73c142-2,11144557\r\nSALD_031308,SALD,SALD,61.0,M,sub-031308_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031308/anat/sub-031308_T1w.nii.gz,ef0ddc0e71658b5ab35cc64466a6c4ee-2,12038810\r\nSALD_031310,SALD,SALD,60.0,F,sub-031310_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031310/anat/sub-031310_T1w.nii.gz,3a6f9e273d34df03dd980df773289f35-2,10396462\r\nSALD_031312,SALD,SALD,74.0,M,sub-031312_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031312/anat/sub-031312_T1w.nii.gz,d4ea575f50dcf698a18da1553cd3f3a9-2,11390093\r\nSALD_031313,SALD,SALD,73.0,M,sub-031313_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031313/anat/sub-031313_T1w.nii.gz,cbffdd4ca9329c6e8eaff1bb56afd662-2,12049184\r\nSALD_031316,SALD,SALD,74.0,F,sub-031316_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031316/anat/sub-031316_T1w.nii.gz,5e7deedcc9520265210f86c5aad0e6e2-2,9942329\r\nSALD_031318,SALD,SALD,76.0,M,sub-031318_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031318/anat/sub-031318_T1w.nii.gz,51ba36070f274d36217463e1159f8aa6-2,10927125\r\nSALD_031319,SALD,SALD,77.0,F,sub-031319_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031319/anat/sub-031319_T1w.nii.gz,00ba8144f9e585590575a74266298d4d-2,10434522\r\nSALD_031320,SALD,SALD,74.0,F,sub-031320_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031320/anat/sub-031320_T1w.nii.gz,111f24ef4877d1185d08da6304cc79d8-2,10281805\r\nSALD_031322,SALD,SALD,80.0,M,sub-031322_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031322/anat/sub-031322_T1w.nii.gz,fe76b788a50f4beb0fe30a299a226b16-2,11263118\r\nSALD_031323,SALD,SALD,76.0,F,sub-031323_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031323/anat/sub-031323_T1w.nii.gz,7c9de9220a8d0f627f344429fd214b4d-2,9696669\r\nSALD_031327,SALD,SALD,63.0,F,sub-031327_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031327/anat/sub-031327_T1w.nii.gz,7522e8c49e514d1cc781f47ab6be9e65-2,10956075\r\nSALD_031332,SALD,SALD,73.0,M,sub-031332_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031332/anat/sub-031332_T1w.nii.gz,25527823af74c836c36d412b8d2e134f-2,11339418\r\nSALD_031337,SALD,SALD,61.0,F,sub-031337_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031337/anat/sub-031337_T1w.nii.gz,1b1fe9284bbf93270e71852f81c2d63e-2,10646241\r\nSALD_031340,SALD,SALD,74.0,M,sub-031340_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031340/anat/sub-031340_T1w.nii.gz,0c0b6d78d0edd01f22e968f3c18d1f96-2,9871170\r\nSALD_031342,SALD,SALD,74.0,F,sub-031342_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031342/anat/sub-031342_T1w.nii.gz,61671be8914f6fe96152c5e6a2f1e39d-2,10791930\r\nSALD_031343,SALD,SALD,70.0,M,sub-031343_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031343/anat/sub-031343_T1w.nii.gz,30a838c8d6aefbdcbd928078c6b76e2b-2,11286807\r\nSALD_031346,SALD,SALD,58.0,F,sub-031346_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031346/anat/sub-031346_T1w.nii.gz,40c5a292cc7829e38b097e293a6646e5-2,10459957\r\nSALD_031348,SALD,SALD,60.0,F,sub-031348_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031348/anat/sub-031348_T1w.nii.gz,8dd26417d8145e297cb558ef861f1266-2,10898179\r\nSALD_031350,SALD,SALD,65.0,M,sub-031350_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031350/anat/sub-031350_T1w.nii.gz,07e8f5e352ab47c74a68c7d8bfc63888-2,11480349\r\nSALD_031351,SALD,SALD,52.0,F,sub-031351_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031351/anat/sub-031351_T1w.nii.gz,ab45e557940782f65d115a211877bcaf-2,10743171\r\nSALD_031356,SALD,SALD,50.0,F,sub-031356_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031356/anat/sub-031356_T1w.nii.gz,dfb18a330b34ea953d6445d67370d312-2,10456827\r\nSALD_031361,SALD,SALD,61.0,F,sub-031361_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031361/anat/sub-031361_T1w.nii.gz,3023937beb70b7dfc6e4ae52f70b0216-2,10445772\r\nSALD_031365,SALD,SALD,72.0,F,sub-031365_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031365/anat/sub-031365_T1w.nii.gz,eedb4f5b01649d0c10a984c718856f81-2,9595258\r\nSALD_031366,SALD,SALD,71.0,F,sub-031366_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031366/anat/sub-031366_T1w.nii.gz,27f8db239b405aa5bef4c147b47ed314-2,10488764\r\nSALD_031367,SALD,SALD,60.0,M,sub-031367_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031367/anat/sub-031367_T1w.nii.gz,e8dea33c9e59138330f5257914ebf275-2,10099557\r\nSALD_031368,SALD,SALD,75.0,F,sub-031368_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031368/anat/sub-031368_T1w.nii.gz,682a8b54ce0aeaa38071a1fd3f435907-2,10112662\r\nSALD_031374,SALD,SALD,63.0,F,sub-031374_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031374/anat/sub-031374_T1w.nii.gz,8a1ba1743f35168b895a3e0f9c768126-2,10305563\r\nSALD_031376,SALD,SALD,49.0,F,sub-031376_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031376/anat/sub-031376_T1w.nii.gz,bf6ab70ffaa876c12f3db510e08e517a-2,10409801\r\nSALD_031377,SALD,SALD,47.0,F,sub-031377_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031377/anat/sub-031377_T1w.nii.gz,202d44f246eaae44303a797b91c012ce-2,10291008\r\nSALD_031378,SALD,SALD,62.0,F,sub-031378_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031378/anat/sub-031378_T1w.nii.gz,625a5343ae597851f1a7f3a8edb666e2-2,9983591\r\nSALD_031379,SALD,SALD,49.0,F,sub-031379_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031379/anat/sub-031379_T1w.nii.gz,6ea15c84d62781c13839e838eff8dc0a-2,9960456\r\nSALD_031380,SALD,SALD,73.0,F,sub-031380_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031380/anat/sub-031380_T1w.nii.gz,75e381325a0ed05ca81631c857ab3eae-2,10403837\r\nSALD_031381,SALD,SALD,60.0,F,sub-031381_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031381/anat/sub-031381_T1w.nii.gz,9dc8b6519e25b27795d76701b5444fda-2,11124901\r\nSALD_031384,SALD,SALD,60.0,M,sub-031384_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031384/anat/sub-031384_T1w.nii.gz,e088d4b548986de660619e913231f106-2,11024622\r\nSALD_031385,SALD,SALD,62.0,F,sub-031385_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031385/anat/sub-031385_T1w.nii.gz,a8e75e46baa24edba002313170112295-2,10778217\r\nSALD_031388,SALD,SALD,57.0,M,sub-031388_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031388/anat/sub-031388_T1w.nii.gz,fe049339bd09465a5970efb164736f4b-2,12074212\r\nSALD_031390,SALD,SALD,64.0,M,sub-031390_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031390/anat/sub-031390_T1w.nii.gz,7265c400f4eb449704ce7c89f1849b02-2,12002519\r\nSALD_031396,SALD,SALD,64.0,M,sub-031396_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031396/anat/sub-031396_T1w.nii.gz,5cbce96dd41f73d93f3cfc7c3b604701-2,10808976\r\nSALD_031401,SALD,SALD,73.0,M,sub-031401_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031401/anat/sub-031401_T1w.nii.gz,90f32fb7f52b57e5c0bcab379a02a0ed-2,11161553\r\nSALD_031402,SALD,SALD,57.0,M,sub-031402_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031402/anat/sub-031402_T1w.nii.gz,4fcbb8b8344d09ef52e81b4bc38e7c23-2,11505034\r\nSALD_031403,SALD,SALD,75.0,F,sub-031403_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031403/anat/sub-031403_T1w.nii.gz,fcf25b80009bea51b7820b72b2743652-2,10604113\r\nSALD_031404,SALD,SALD,75.0,F,sub-031404_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031404/anat/sub-031404_T1w.nii.gz,dbfe903981539d03b15ce859d3348df6-2,9775376\r\nSALD_031406,SALD,SALD,60.0,F,sub-031406_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031406/anat/sub-031406_T1w.nii.gz,044399ce53a3a5607e65f35e17fca4c9-2,10588142\r\nSALD_031407,SALD,SALD,62.0,M,sub-031407_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031407/anat/sub-031407_T1w.nii.gz,996f6c79bb46d4c2838b43d8cabca770-2,11233043\r\nSALD_031409,SALD,SALD,65.0,F,sub-031409_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031409/anat/sub-031409_T1w.nii.gz,ad161d8a60ec6eea365e84836bcf0424-2,10232212\r\nSALD_031410,SALD,SALD,51.0,F,sub-031410_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031410/anat/sub-031410_T1w.nii.gz,3d0e2a80e6c3c76debf1cad655aea7d6-2,9878632\r\nSALD_031412,SALD,SALD,63.0,M,sub-031412_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031412/anat/sub-031412_T1w.nii.gz,fcde2a30a1680d55d46906da825618a8-2,9356035\r\nSALD_031414,SALD,SALD,63.0,F,sub-031414_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031414/anat/sub-031414_T1w.nii.gz,3d9f98cdf262da6b4d045c43f352b85a-2,11161374\r\nSALD_031415,SALD,SALD,50.0,F,sub-031415_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031415/anat/sub-031415_T1w.nii.gz,47d421dfe9931ef2d3e55eba3f36698f-2,10092867\r\nSALD_031417,SALD,SALD,56.0,M,sub-031417_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031417/anat/sub-031417_T1w.nii.gz,8b82ee57d66f31720a9dad6565836b06-2,11131614\r\nSALD_031418,SALD,SALD,71.0,F,sub-031418_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031418/anat/sub-031418_T1w.nii.gz,9964eb2113555b86b35a8905f409fa5e-2,10778999\r\nSALD_031420,SALD,SALD,55.0,M,sub-031420_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031420/anat/sub-031420_T1w.nii.gz,48c5812edb1943961ac81083b59d03db-2,10792797\r\nSALD_031424,SALD,SALD,74.0,F,sub-031424_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031424/anat/sub-031424_T1w.nii.gz,08bcd462c775e9777822862b3c67ef84-2,10600126\r\nSALD_031425,SALD,SALD,59.0,F,sub-031425_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031425/anat/sub-031425_T1w.nii.gz,35efec93a05fe9e770a110ac332bd0a3-2,11566324\r\nSALD_031427,SALD,SALD,77.0,M,sub-031427_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031427/anat/sub-031427_T1w.nii.gz,f1d210b24fefb8b9943cffb4572d934c-2,11126305\r\nSALD_031428,SALD,SALD,73.0,F,sub-031428_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031428/anat/sub-031428_T1w.nii.gz,1310a5e095d9edeb135e4df5dc4e8149-2,11242347\r\nSALD_031429,SALD,SALD,78.0,M,sub-031429_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031429/anat/sub-031429_T1w.nii.gz,1aba2aa85f927f567221f9df7a0b6409-2,11460420\r\nSALD_031430,SALD,SALD,77.0,F,sub-031430_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031430/anat/sub-031430_T1w.nii.gz,50000e1df6486f06faf560344a5a89ce-2,10868518\r\nSALD_031431,SALD,SALD,58.0,M,sub-031431_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031431/anat/sub-031431_T1w.nii.gz,d2788dacbf25eb51034c7b2508e296fd-2,11471800\r\nSALD_031433,SALD,SALD,59.0,M,sub-031433_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031433/anat/sub-031433_T1w.nii.gz,4b74b1231dcc7464ac8229fbf50a6422-2,12233937\r\nSALD_031434,SALD,SALD,59.0,F,sub-031434_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031434/anat/sub-031434_T1w.nii.gz,c485faeb525133caa4e8d55f2fe2e248-2,9617525\r\nSALD_031435,SALD,SALD,66.0,F,sub-031435_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031435/anat/sub-031435_T1w.nii.gz,00709020ca1e80e3f70d5b220e2604ab-2,10722002\r\nSALD_031437,SALD,SALD,30.0,F,sub-031437_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031437/anat/sub-031437_T1w.nii.gz,1ea4fcef9199caea64341f0b127fe5d2-2,9881514\r\nSALD_031439,SALD,SALD,49.0,F,sub-031439_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031439/anat/sub-031439_T1w.nii.gz,610d3ffae4bac6c7a1adc3b68e87686a-2,11555506\r\nSALD_031441,SALD,SALD,75.0,F,sub-031441_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031441/anat/sub-031441_T1w.nii.gz,f55b14d5e94d26ca35e907b222641e43-2,9834888\r\nSALD_031442,SALD,SALD,48.0,F,sub-031442_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031442/anat/sub-031442_T1w.nii.gz,7972737591020998080c48c874d57953-2,10546956\r\nSALD_031445,SALD,SALD,68.0,F,sub-031445_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031445/anat/sub-031445_T1w.nii.gz,a36053dee8cf3efd76ead7e94a689629-2,10284699\r\nSALD_031446,SALD,SALD,57.0,F,sub-031446_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031446/anat/sub-031446_T1w.nii.gz,65678f5f9e991ed3e3afc3d3613b80c9-2,11193982\r\nSALD_031447,SALD,SALD,57.0,M,sub-031447_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031447/anat/sub-031447_T1w.nii.gz,65f649770e0d8748180d9dec481c34d4-2,11404443\r\nSALD_031450,SALD,SALD,66.0,M,sub-031450_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031450/anat/sub-031450_T1w.nii.gz,e2dd03ce93363d42791a1506034104e9-2,11367643\r\nSALD_031453,SALD,SALD,49.0,F,sub-031453_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031453/anat/sub-031453_T1w.nii.gz,255343becdb4a8ed2c1929f87b201271-2,9608453\r\nSALD_031455,SALD,SALD,59.0,F,sub-031455_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031455/anat/sub-031455_T1w.nii.gz,918310c8053e1242ebc638780f6f69e4-2,11302519\r\nSALD_031457,SALD,SALD,34.0,F,sub-031457_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031457/anat/sub-031457_T1w.nii.gz,51952765ae6cfb5bff7c4b7c19037edd-2,10356811\r\nSALD_031458,SALD,SALD,44.0,F,sub-031458_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031458/anat/sub-031458_T1w.nii.gz,05645772351df10fd0f44c3a920b0ddc-2,10698392\r\nSALD_031459,SALD,SALD,66.0,F,sub-031459_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031459/anat/sub-031459_T1w.nii.gz,2185222e74f14c1d70d4f0b71d690d71-2,11056746\r\nSALD_031460,SALD,SALD,78.0,F,sub-031460_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031460/anat/sub-031460_T1w.nii.gz,42cbf8560cb565003d0d9897eaa1b3d6-2,10360481\r\nSALD_031462,SALD,SALD,69.0,F,sub-031462_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031462/anat/sub-031462_T1w.nii.gz,343cb077397e6acb4f9cc07650930fa8-2,9698164\r\nSALD_031463,SALD,SALD,51.0,F,sub-031463_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031463/anat/sub-031463_T1w.nii.gz,a37d5776eda77021c3978148a603c258-2,10620275\r\nSALD_031467,SALD,SALD,24.0,M,sub-031467_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031467/anat/sub-031467_T1w.nii.gz,2417d9ed3728eca5f2ffc91661f5dcc0-2,10818202\r\nSALD_031468,SALD,SALD,22.0,M,sub-031468_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031468/anat/sub-031468_T1w.nii.gz,43dcf6f1c0bc2c77fee04041e0c13feb-2,11575751\r\nSALD_031469,SALD,SALD,22.0,M,sub-031469_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031469/anat/sub-031469_T1w.nii.gz,6cf1ad6a62c1fbd416aaf91d02030483-2,11372532\r\nSALD_031472,SALD,SALD,23.0,M,sub-031472_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031472/anat/sub-031472_T1w.nii.gz,c4dd7316ca750d766babb93d1b62e0ac-2,10843442\r\nSALD_031473,SALD,SALD,20.0,M,sub-031473_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031473/anat/sub-031473_T1w.nii.gz,8306f5ba564b0c71022e369fb2b18813-2,11826546\r\nSALD_031480,SALD,SALD,45.0,F,sub-031480_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031480/anat/sub-031480_T1w.nii.gz,ab17352d8180fc0b674abebfd3a1a8bf-2,10401514\r\nSALD_031489,SALD,SALD,41.0,F,sub-031489_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031489/anat/sub-031489_T1w.nii.gz,dd5028cd5b8b69b14a1615ae04b383ff-2,9616897\r\nSALD_031493,SALD,SALD,35.0,M,sub-031493_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031493/anat/sub-031493_T1w.nii.gz,aadc248c3eee5b4633734b2c94822a78-2,11133910\r\nSALD_031495,SALD,SALD,45.0,F,sub-031495_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031495/anat/sub-031495_T1w.nii.gz,e08f4963ca5c07c6d48d489ce9114d66-2,10780446\r\nSALD_031499,SALD,SALD,52.0,M,sub-031499_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031499/anat/sub-031499_T1w.nii.gz,9fa431a9a07cfa9c8a4437273090eee1-2,11186012\r\nSALD_031500,SALD,SALD,48.0,M,sub-031500_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031500/anat/sub-031500_T1w.nii.gz,896b03ac9d26eeb0b5220058b0c877c7-2,10512640\r\nSALD_031506,SALD,SALD,27.0,F,sub-031506_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031506/anat/sub-031506_T1w.nii.gz,bd80c8cf9d1d75ae9425c55ef25ed05e-2,10098702\r\nSALD_031507,SALD,SALD,50.0,F,sub-031507_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031507/anat/sub-031507_T1w.nii.gz,fa97e60bc67f7d693defd5a15c38dd33-2,11185664\r\nSALD_031508,SALD,SALD,33.0,M,sub-031508_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031508/anat/sub-031508_T1w.nii.gz,0fa1a217220ef1bfec121419473a3162-2,10042229\r\nSALD_031514,SALD,SALD,50.0,M,sub-031514_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031514/anat/sub-031514_T1w.nii.gz,c0ff96f108e3b726d45dc6d8a8de5b70-2,10706516\r\nSALD_031516,SALD,SALD,29.0,M,sub-031516_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031516/anat/sub-031516_T1w.nii.gz,ebd36b69262a44351ab1e45d2fa62083-2,11486250\r\nSALD_031517,SALD,SALD,31.0,M,sub-031517_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031517/anat/sub-031517_T1w.nii.gz,108b08285178311e8d54101ab625c964-2,12197040\r\nSALD_031519,SALD,SALD,30.0,F,sub-031519_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031519/anat/sub-031519_T1w.nii.gz,0a7303dca3969ce842c94449aa6d9316-2,10316687\r\nSALD_031520,SALD,SALD,30.0,M,sub-031520_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031520/anat/sub-031520_T1w.nii.gz,40dec881de8c62423536cce8de2c3d9c-2,11245528\r\nSALD_031521,SALD,SALD,29.0,M,sub-031521_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031521/anat/sub-031521_T1w.nii.gz,01821293a6c2e961e39b986afedc6310-2,11147275\r\nSALD_031522,SALD,SALD,37.0,F,sub-031522_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031522/anat/sub-031522_T1w.nii.gz,b1b4ad380ef213d28d149ffc1586525c-2,10884806\r\nSALD_031529,SALD,SALD,30.0,M,sub-031529_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031529/anat/sub-031529_T1w.nii.gz,095c0299c17e8948c91a34b67377bd22-2,11904167\r\nSALD_031534,SALD,SALD,32.0,F,sub-031534_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031534/anat/sub-031534_T1w.nii.gz,2d0f7f15b7326c397884ded262b0034b-2,10693367\r\nSALD_031535,SALD,SALD,42.0,F,sub-031535_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031535/anat/sub-031535_T1w.nii.gz,0d4d95698c830fcbeb3bf48eaeb4e3e6-2,10854622\r\nSALD_031536,SALD,SALD,47.0,M,sub-031536_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031536/anat/sub-031536_T1w.nii.gz,db7453dbfb98fe0762a3d0af9e4c86db-2,10266437\r\nSALD_031541,SALD,SALD,44.0,M,sub-031541_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031541/anat/sub-031541_T1w.nii.gz,fe91d8f6d83c8e0130c88c012cc79701-2,10398720\r\nSALD_031542,SALD,SALD,44.0,M,sub-031542_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031542/anat/sub-031542_T1w.nii.gz,27f682d1fd740e181728bd5f5ef615e5-2,10879770\r\nSALD_031546,SALD,SALD,38.0,M,sub-031546_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031546/anat/sub-031546_T1w.nii.gz,f90f3014f0cc826521b4ba57925cfb83-2,11408960\r\nSALD_031547,SALD,SALD,38.0,F,sub-031547_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031547/anat/sub-031547_T1w.nii.gz,ff046e1b73bd6a61153fa94b4eebfc40-2,10326665\r\nSALD_031548,SALD,SALD,42.0,F,sub-031548_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031548/anat/sub-031548_T1w.nii.gz,acaef0a5867f8e3599863e0b7014876f-2,10266904\r\nSALD_031550,SALD,SALD,48.0,M,sub-031550_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031550/anat/sub-031550_T1w.nii.gz,0d863ec3fee3f6755e9b45a3b2b58949-2,11788702\r\nSALD_031553,SALD,SALD,22.0,F,sub-031553_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031553/anat/sub-031553_T1w.nii.gz,61bc71aed0f6a618e4f00a9d9960225c-2,10652187\r\nSALD_031555,SALD,SALD,39.0,M,sub-031555_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031555/anat/sub-031555_T1w.nii.gz,9f93db50f5f5f0c688a290f7a550bebe-2,11016627\r\nSALD_031558,SALD,SALD,46.0,F,sub-031558_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031558/anat/sub-031558_T1w.nii.gz,60fe0625391017ff9eb913bda8f41913-2,10178175\r\nSALD_031561,SALD,SALD,40.0,M,sub-031561_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031561/anat/sub-031561_T1w.nii.gz,49b4c099064671be652a844509a83e3b-2,10782031\r\nSALD_031562,SALD,SALD,21.0,F,sub-031562_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031562/anat/sub-031562_T1w.nii.gz,3e477a3a828628a3b0ce5d35192ae84a-2,10151530\r\nSALD_031563,SALD,SALD,21.0,F,sub-031563_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031563/anat/sub-031563_T1w.nii.gz,3f35d6e5940432c1297bfe711c161178-2,11128846\r\nSALD_031577,SALD,SALD,49.0,F,sub-031577_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031577/anat/sub-031577_T1w.nii.gz,e7368a7be8478addf03829c31130e81d-2,10662001\r\nSALD_031578,SALD,SALD,37.0,M,sub-031578_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031578/anat/sub-031578_T1w.nii.gz,c3b6b6ef44f25fed4c6949e7a1cee397-2,10521600\r\nSALD_031579,SALD,SALD,20.0,M,sub-031579_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031579/anat/sub-031579_T1w.nii.gz,0f0415af77fccdcf4d5548af063b13c1-2,10769340\r\nSALD_031581,SALD,SALD,25.0,F,sub-031581_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031581/anat/sub-031581_T1w.nii.gz,36702acf041b69002eb77b14745704b9-2,11034560\r\nSALD_031586,SALD,SALD,48.0,F,sub-031586_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031586/anat/sub-031586_T1w.nii.gz,2fb8f110c92c0a1de9549cc16ff0a923-2,11035214\r\nSALD_031596,SALD,SALD,24.0,F,sub-031596_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031596/anat/sub-031596_T1w.nii.gz,25a348d83c6897d82072fb4a91696fad-2,10724236\r\nSALD_031598,SALD,SALD,39.0,F,sub-031598_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031598/anat/sub-031598_T1w.nii.gz,250ab604c8a5505a009de7b3b16f61d7-2,10877668\r\nSALD_031621,SALD,SALD,24.0,M,sub-031621_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031621/anat/sub-031621_T1w.nii.gz,4a98ec3d45ac18afa073f3c8c399d87c-2,10948393\r\nSALD_031628,SALD,SALD,22.0,F,sub-031628_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031628/anat/sub-031628_T1w.nii.gz,303e5bab75224191217517b4fa8e6851-2,11589765\r\nSALD_031629,SALD,SALD,21.0,F,sub-031629_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031629/anat/sub-031629_T1w.nii.gz,a212b0849aa7a00fdb262f5dbf4547cd-2,10195188\r\nSALD_031632,SALD,SALD,24.0,F,sub-031632_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031632/anat/sub-031632_T1w.nii.gz,d5911f96328cbf2b7105ef01d4aff667-2,10291947\r\nSALD_031634,SALD,SALD,38.0,F,sub-031634_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031634/anat/sub-031634_T1w.nii.gz,cfea0d5222aebf9e71151dd460d101d8-2,11794353\r\nSALD_031636,SALD,SALD,53.0,M,sub-031636_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031636/anat/sub-031636_T1w.nii.gz,50fffab1b1378f4805cd8c616ba9e7b8-2,9320512\r\nSALD_031638,SALD,SALD,46.0,F,sub-031638_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031638/anat/sub-031638_T1w.nii.gz,f4082fcc7b92bc8c085d7cdd8dbf9ff3-2,11035669\r\nSALD_031639,SALD,SALD,42.0,F,sub-031639_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031639/anat/sub-031639_T1w.nii.gz,ac56f391249b0417e3091e2d5cba28d8-2,11375752\r\nSALD_031642,SALD,SALD,56.0,M,sub-031642_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031642/anat/sub-031642_T1w.nii.gz,c548f9a6689b1ab15976ec689d665a79-2,10292065\r\nSALD_031644,SALD,SALD,35.0,M,sub-031644_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031644/anat/sub-031644_T1w.nii.gz,dba3b1abf204824060095cfe58f5cb69-2,10844246\r\nSALD_031645,SALD,SALD,64.0,F,sub-031645_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031645/anat/sub-031645_T1w.nii.gz,f35833a6be166016fcbe967ac331be1c-2,11925954\r\nSALD_031647,SALD,SALD,36.0,F,sub-031647_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031647/anat/sub-031647_T1w.nii.gz,f76d5493af7e2c1943ff8d093b21ce0f-2,11011563\r\nSALD_031649,SALD,SALD,59.0,M,sub-031649_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031649/anat/sub-031649_T1w.nii.gz,b228ff2642125f6fb74491aee837517d-2,10433749\r\nSALD_031651,SALD,SALD,34.0,M,sub-031651_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031651/anat/sub-031651_T1w.nii.gz,40e20568727713594b555a0207ec71d2-2,11578538\r\nSALD_031653,SALD,SALD,34.0,M,sub-031653_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031653/anat/sub-031653_T1w.nii.gz,301d5061afdba3426af54ab6303cd502-2,9417134\r\nSALD_031654,SALD,SALD,34.0,F,sub-031654_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031654/anat/sub-031654_T1w.nii.gz,5f92ce213969e42651cb072f230e9e2a-2,10825315\r\nSALD_031656,SALD,SALD,34.0,F,sub-031656_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031656/anat/sub-031656_T1w.nii.gz,b5008a497d2748a635c2a99ba37b45b3-2,12387485\r\nSALD_031657,SALD,SALD,34.0,F,sub-031657_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031657/anat/sub-031657_T1w.nii.gz,12194b524357b8e57b1b084593aaec60-2,11076476\r\nSALD_031658,SALD,SALD,53.0,M,sub-031658_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031658/anat/sub-031658_T1w.nii.gz,57ce473ac46c863416075c6583bdd9f2-2,12104214\r\nSALD_031659,SALD,SALD,65.0,M,sub-031659_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031659/anat/sub-031659_T1w.nii.gz,5729b4207e1e10103bddebe0db6f23b8-2,10837415\r\nSALD_031660,SALD,SALD,42.0,F,sub-031660_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031660/anat/sub-031660_T1w.nii.gz,f3c2352d6f0c898abb00bacaac1f579d-2,11569044\r\nSALD_031661,SALD,SALD,59.0,M,sub-031661_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031661/anat/sub-031661_T1w.nii.gz,b0e6afcdeceb1c9744b54415ec61e886-2,10166317\r\nSALD_031662,SALD,SALD,44.0,F,sub-031662_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031662/anat/sub-031662_T1w.nii.gz,8378a7bbf07dd1b302e53da11ffba424-2,11244696\r\nSALD_031663,SALD,SALD,48.0,F,sub-031663_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031663/anat/sub-031663_T1w.nii.gz,407446953b1001fae017c68842d5309f-2,12426258\r\nSALD_031664,SALD,SALD,61.0,M,sub-031664_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031664/anat/sub-031664_T1w.nii.gz,64d7590580d9e7aa50322f116a3f851a-2,9419744\r\nSALD_031666,SALD,SALD,52.0,F,sub-031666_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031666/anat/sub-031666_T1w.nii.gz,a71648490cbfbfdca3d60b66e30c1039-2,10537471\r\nSALD_031670,SALD,SALD,60.0,M,sub-031670_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031670/anat/sub-031670_T1w.nii.gz,b6163a6c16348b7273aa5c399c4d85cb-2,10719042\r\nSALD_031674,SALD,SALD,37.0,F,sub-031674_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031674/anat/sub-031674_T1w.nii.gz,deaf06632114cfc128d63aa71ffa8fc2-2,11982296\r\nSALD_031676,SALD,SALD,33.0,F,sub-031676_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031676/anat/sub-031676_T1w.nii.gz,ad17e0fbcc6aa812f3b7951a70f46e7b-2,10913157\r\nSALD_031678,SALD,SALD,35.0,M,sub-031678_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031678/anat/sub-031678_T1w.nii.gz,ee394539d2896bf69db9c45b9fe2cd5f-2,10383452\r\nSALD_031680,SALD,SALD,33.0,M,sub-031680_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031680/anat/sub-031680_T1w.nii.gz,8133c3207adedbe2b63715b559f3d4a7-2,10636417\r\nSALD_031682,SALD,SALD,44.0,F,sub-031682_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031682/anat/sub-031682_T1w.nii.gz,c8a7db8b3e92c7ff9cb828afffa04efc-2,11048762\r\nSALD_031684,SALD,SALD,59.0,M,sub-031684_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031684/anat/sub-031684_T1w.nii.gz,188f282bf8094a91d7b3cf471ff103e1-2,11399289\r\nSALD_031687,SALD,SALD,42.0,M,sub-031687_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031687/anat/sub-031687_T1w.nii.gz,d8e46151e9c21e0accd3f23d628defec-2,9698369\r\nSALD_031688,SALD,SALD,34.0,F,sub-031688_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031688/anat/sub-031688_T1w.nii.gz,188c9cd068cc89ab1a1eb2f13fa7c9b3-2,11300353\r\nSALD_031690,SALD,SALD,33.0,F,sub-031690_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031690/anat/sub-031690_T1w.nii.gz,01c0c27551da10530ddeb20e18ef2e79-2,12003527\r\nSALD_031692,SALD,SALD,35.0,F,sub-031692_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031692/anat/sub-031692_T1w.nii.gz,404174d13c1126d1059b88d5347da42a-2,11488257\r\nSALD_031693,SALD,SALD,34.0,M,sub-031693_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031693/anat/sub-031693_T1w.nii.gz,f1d2ad9ceabc939d9811be15d05e017f-2,10195006\r\nSALD_031694,SALD,SALD,32.0,F,sub-031694_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031694/anat/sub-031694_T1w.nii.gz,5a5d0e71a04c23dd72de1a5caa5ace23-2,12190894\r\nSALD_031704,SALD,SALD,42.0,M,sub-031704_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031704/anat/sub-031704_T1w.nii.gz,6d334be5fcbc909110afb3e2d30866ed-2,10594512\r\nSALD_031711,SALD,SALD,56.0,F,sub-031711_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031711/anat/sub-031711_T1w.nii.gz,7dfab3793cb91dab6ad4c46f2096ab1c-2,9889608\r\nSALD_031713,SALD,SALD,54.0,M,sub-031713_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031713/anat/sub-031713_T1w.nii.gz,fb131d3fd15edf83a435cea883bd6e73-2,10370599\r\nSALD_031715,SALD,SALD,25.0,M,sub-031715_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031715/anat/sub-031715_T1w.nii.gz,5aa0e88c1a6927fff481fb038d46ff97-2,10684645\r\nSALD_031718,SALD,SALD,24.0,M,sub-031718_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031718/anat/sub-031718_T1w.nii.gz,f6fa3d155feaf9c1a34db919d089670c-2,11471076\r\nSALD_031721,SALD,SALD,24.0,F,sub-031721_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031721/anat/sub-031721_T1w.nii.gz,b1327fbdbdfe186dfa30fdebc439a632-2,11198249\r\nSALD_031722,SALD,SALD,24.0,F,sub-031722_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031722/anat/sub-031722_T1w.nii.gz,af4748f16dfeb43ee953204a1275b56c-2,10251339\r\nSALD_031729,SALD,SALD,30.0,M,sub-031729_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031729/anat/sub-031729_T1w.nii.gz,4b285af3b83168a7afe8283029143e2f-2,11437737\r\nSALD_031730,SALD,SALD,25.0,F,sub-031730_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031730/anat/sub-031730_T1w.nii.gz,5d42aab611cf371e13e65bef78955104-2,10527654\r\nSALD_031731,SALD,SALD,24.0,F,sub-031731_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031731/anat/sub-031731_T1w.nii.gz,190422239e9ab413bb2b969bd6f55c68-2,10728864\r\nSALD_031732,SALD,SALD,24.0,F,sub-031732_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031732/anat/sub-031732_T1w.nii.gz,3d0da473772c158182535d63cab43495-2,10128982\r\nSALD_031734,SALD,SALD,25.0,F,sub-031734_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031734/anat/sub-031734_T1w.nii.gz,6208d73a9500b8183c97a13ec7341410-2,10581031\r\nSALD_031735,SALD,SALD,24.0,F,sub-031735_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031735/anat/sub-031735_T1w.nii.gz,d89a3b9a4b60ee86f60028837392a48c-2,11227069\r\nSALD_031740,SALD,SALD,26.0,F,sub-031740_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031740/anat/sub-031740_T1w.nii.gz,d9ab28c0ef24bcfd7935560b4fc0ce56-2,10834381\r\nSALD_031744,SALD,SALD,25.0,M,sub-031744_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031744/anat/sub-031744_T1w.nii.gz,7251ea5b8048edbf019d566a0207e899-2,12025787\r\nSALD_031746,SALD,SALD,25.0,M,sub-031746_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031746/anat/sub-031746_T1w.nii.gz,01f11c80527c298c191f11267b45ae9e-2,10105893\r\nSALD_031747,SALD,SALD,25.0,M,sub-031747_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031747/anat/sub-031747_T1w.nii.gz,a55447dac196926920b4787ec452e96c-2,10514102\r\nSALD_031751,SALD,SALD,27.0,F,sub-031751_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031751/anat/sub-031751_T1w.nii.gz,57169dc7fee731d24204f148b4314152-2,10593713\r\nSALD_031752,SALD,SALD,34.0,M,sub-031752_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031752/anat/sub-031752_T1w.nii.gz,e4aead1decb5431057af94bd06241e36-2,11786048\r\nSALD_031759,SALD,SALD,25.0,M,sub-031759_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031759/anat/sub-031759_T1w.nii.gz,720f9eb94e1e467ba062b88ab6ac5ff2-2,11989551\r\nSALD_031760,SALD,SALD,23.0,M,sub-031760_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/SALD/RawData_BIDS/sub-031760/anat/sub-031760_T1w.nii.gz,afb98871e1defbbb1acf4c7338c6151c-2,10402383\r\n"
IXI_TRAIN_CSV = "scan_id,chronological_age\r\nIXI002,35.8001\r\nIXI012,38.7817\r\nIXI013,46.7105\r\nIXI014,34.2368\r\nIXI015,24.2847\r\nIXI016,55.1677\r\nIXI017,29.0924\r\nIXI019,58.6585\r\nIXI020,39.4661\r\nIXI021,21.5661\r\nIXI022,30.6721\r\nIXI023,33.0787\r\nIXI024,37.6975\r\nIXI025,29.7796\r\nIXI026,34.4778\r\nIXI027,30.4175\r\nIXI028,74.026\r\nIXI029,59.2225\r\nIXI030,31.4606\r\nIXI031,31.1348\r\nIXI033,24.7639\r\nIXI034,24.3422\r\nIXI036,23.5373\r\nIXI037,37.2101\r\nIXI038,21.3826\r\nIXI039,23.3484\r\nIXI040,44.0931\r\nIXI041,27.3895\r\nIXI042,27.822\r\nIXI043,22.6475\r\nIXI044,44.8515\r\nIXI045,40.0356\r\nIXI046,37.7331\r\nIXI048,50.6502\r\nIXI049,31.8549\r\nIXI050,63.1869\r\nIXI051,25.7413\r\nIXI052,33.7632\r\nIXI053,53.306\r\nIXI054,41.7522\r\nIXI055,49.4045\r\nIXI056,25.9384\r\nIXI057,28.1588\r\nIXI058,29.2786\r\nIXI059,34.1383\r\nIXI060,30.6995\r\nIXI061,59.0034\r\nIXI062,36.2163\r\nIXI063,41.1225\r\nIXI064,20.0712\r\nIXI065,23.7153\r\nIXI066,46.1684\r\nIXI067,24.6899\r\nIXI068,24.5695\r\nIXI069,26.2724\r\nIXI070,20.6982\r\nIXI071,37.0021\r\nIXI072,68.6023\r\nIXI073,46.3655\r\nIXI075,35.9918\r\nIXI076,55.8275\r\nIXI077,36.4791\r\nIXI078,28.5065\r\nIXI079,52.5339\r\nIXI080,21.1992\r\nIXI083,30.8939\r\nIXI084,38.2341\r\nIXI085,31.8494\r\nIXI086,30.0698\r\nIXI087,25.6619\r\nIXI089,49.4045\r\nIXI090,41.8207\r\nIXI091,28.6133\r\nIXI092,33.2402\r\nIXI093,26.8775\r\nIXI094,24.8542\r\nIXI095,24.8953\r\nIXI096,26.924\r\nIXI097,35.0226\r\nIXI098,29.7303\r\nIXI099,51.7536\r\nIXI100,41.5113\r\nIXI101,45.7741\r\nIXI102,24.6215\r\nIXI103,31.8084\r\nIXI104,26.9843\r\nIXI105,38.9021\r\nIXI106,36.6708\r\nIXI107,31.896\r\nIXI108,43.1622\r\nIXI109,36.4682\r\nIXI110,37.7659\r\nIXI111,25.1855\r\nIXI112,23.6988\r\nIXI113,32.3614\r\nIXI114,65.41\r\nIXI115,72.1314\r\nIXI116,23.8467\r\nIXI118,25.9274\r\nIXI119,32.1697\r\nIXI120,53.6701\r\nIXI121,35.0719\r\nIXI122,22.9733\r\nIXI123,46.5791\r\nIXI126,57.2211\r\nIXI127,29.4976\r\nIXI128,27.3265\r\nIXI129,23.039\r\nIXI130,38.7734\r\nIXI131,45.2293\r\nIXI132,58.7159\r\nIXI134,47.3539\r\nIXI135,29.0048\r\nIXI136,35.8056\r\nIXI137,41.2293\r\nIXI138,33.1389\r\nIXI139,32.9363\r\nIXI140,27.8357\r\nIXI141,46.0068\r\nIXI142,48.2546\r\nIXI143,31.9562\r\nIXI144,29.358\r\nIXI145,49.3854\r\nIXI146,33.5934\r\nIXI148,40.5558\r\nIXI150,27.0144\r\nIXI151,45.3881\r\nIXI153,22.4312\r\nIXI154,24.3422\r\nIXI156,59.3812\r\nIXI157,54.4257\r\nIXI158,75.6331\r\nIXI159,24.1424\r\nIXI160,39.3977\r\nIXI161,65.1143\r\nIXI162,50.5243\r\nIXI163,48.052\r\nIXI164,64.4682\r\nIXI165,24.3751\r\nIXI166,41.2019\r\nIXI167,67.5346\r\nIXI168,43.8248\r\nIXI169,58.768\r\nIXI170,31.5921\r\nIXI172,74.9897\r\nIXI173,29.5743\r\nIXI174,63.0965\r\nIXI175,72.2656\r\nIXI176,24.8542\r\nIXI177,34.2888\r\nIXI178,25.4073\r\nIXI179,30.1437\r\nIXI180,42.152\r\nIXI181,45.6975\r\nIXI182,60.9747\r\nIXI183,30.1766\r\nIXI184,30.1191\r\nIXI185,83.4579\r\nIXI186,61.1526\r\nIXI188,44.0246\r\nIXI189,41.7194\r\nIXI191,44.2053\r\nIXI192,38.9377\r\nIXI193,55.2389\r\nIXI194,48.3313\r\nIXI195,37.8289\r\nIXI196,29.1143\r\nIXI197,60.4627\r\nIXI198,45.2567\r\nIXI199,63.6632\r\nIXI200,49.9192\r\nIXI201,22.4148\r\nIXI202,20.167\r\nIXI204,74.6366\r\nIXI205,45.3799\r\nIXI206,50.0233\r\nIXI207,40.3614\r\nIXI208,33.9959\r\nIXI209,32.6434\r\nIXI210,69.3251\r\nIXI211,55.1184\r\nIXI212,59.1294\r\nIXI213,54.6639\r\nIXI214,24.3121\r\nIXI216,73.0157\r\nIXI217,57.2101\r\nIXI218,35.4059\r\nIXI219,53.0815\r\nIXI221,48.5476\r\nIXI222,36.0602\r\nIXI223,63.2471\r\nIXI224,36.9199\r\nIXI225,47.0144\r\nIXI226,41.9028\r\nIXI227,67.0554\r\nIXI229,72.3504\r\nIXI236,25.7139\r\nIXI237,78.204\r\nIXI239,62.9596\r\nIXI240,39.0363\r\nIXI241,30.0068\r\nIXI242,31.3703\r\nIXI244,49.0431\r\nIXI246,42.2204\r\nIXI247,29.8179\r\nIXI248,44.8761\r\nIXI249,68.2683\r\nIXI250,72.8843\r\nIXI251,80.1697\r\nIXI252,70.5599\r\nIXI253,24.0027\r\nIXI254,25.5825\r\nIXI255,57.7029\r\nIXI256,27.384\r\nIXI257,69.5469\r\nIXI258,22.5708\r\nIXI259,59.7837\r\nIXI260,60.9062\r\nIXI261,34.2067\r\nIXI262,20.9281\r\nIXI263,25.2129\r\nIXI264,44.8816\r\nIXI266,61.065\r\nIXI267,35.447\r\nIXI268,44.9199\r\nIXI269,26.5791\r\nIXI270,61.0157\r\nIXI274,73.5387\r\nIXI275,24.1752\r\nIXI276,20.909\r\nIXI277,61.3361\r\nIXI278,20.2136\r\nIXI279,51.8823\r\nIXI280,42.5079\r\nIXI282,40.0082\r\nIXI284,63.0445\r\nIXI285,40.8131\r\nIXI286,54.2779\r\nIXI287,44.9911\r\nIXI288,78.5845\r\nIXI289,60.8405\r\nIXI295,50.9021\r\nIXI296,63.9726\r\nIXI297,51.2909\r\nIXI298,43.4606\r\nIXI299,53.4073\r\nIXI300,38.2779\r\nIXI302,59.5099\r\nIXI304,46.59\r\nIXI308,41.2047\r\nIXI311,61.588\r\nIXI312,59.1485\r\nIXI313,65.8535\r\nIXI316,56.9391\r\nIXI317,67.7947\r\nIXI318,54.486\r\nIXI319,59.4716\r\nIXI320,29.7385\r\nIXI321,54.7433\r\nIXI324,60.0794\r\nIXI325,52.6653\r\nIXI326,67.7399\r\nIXI327,73.5633\r\nIXI328,61.7851\r\nIXI329,55.655\r\nIXI330,44.4819\r\nIXI334,69.2895\r\nIXI335,69.2758\r\nIXI336,67.7728\r\nIXI338,61.4867\r\nIXI342,48.679\r\nIXI344,62.0945\r\nIXI348,38.5873\r\nIXI350,62.6804\r\nIXI351,56.9199\r\nIXI353,52.6899\r\nIXI354,45.6153\r\nIXI356,64.104\r\nIXI357,63.54\r\nIXI358,62.0616\r\nIXI359,62.0616\r\nIXI360,54.193\r\nIXI361,54.193\r\nIXI362,41.0951\r\nIXI363,60.9199\r\nIXI364,67.1622\r\nIXI365,66.527\r\nIXI367,54.2149\r\nIXI368,54.2149\r\nIXI369,60.5969\r\nIXI370,60.5969\r\nIXI375,41.0595\r\nIXI376,72.5941\r\nIXI377,72.5941\r\nIXI379,66.9076\r\nIXI380,66.9076\r\nIXI381,59.6715\r\nIXI383,74.0123\r\nIXI384,74.0123\r\nIXI385,64.6461\r\nIXI386,52.3012\r\nIXI387,46.4312\r\nIXI389,58.3682\r\nIXI390,41.8891\r\nIXI391,52.5311\r\nIXI392,62.4559\r\nIXI393,60.1013\r\nIXI394,53.7495\r\nIXI396,53.5661\r\nIXI397,51.6578\r\nIXI398,51.6578\r\nIXI399,73.5441\r\nIXI400,55.5811\r\nIXI401,55.5811\r\nIXI402,53.5359\r\nIXI403,53.5359\r\nIXI404,62.7379\r\nIXI405,58.5681\r\nIXI406,58.2834\r\nIXI407,58.2806\r\nIXI408,52.1287\r\nIXI409,70.9514\r\nIXI410,53.4127\r\nIXI411,53.4127\r\nIXI412,58.5681\r\nIXI413,59.8357\r\nIXI414,59.7426\r\nIXI415,72.3258\r\nIXI416,72.1451\r\nIXI417,59.8823\r\nIXI418,59.7426\r\nIXI419,73.1828\r\nIXI420,73.1828\r\nIXI422,37.0979\r\nIXI428,55.54\r\nIXI429,55.54\r\nIXI431,58.7817\r\nIXI432,57.7385\r\nIXI436,28.3477\r\nIXI437,55.5975\r\nIXI438,63.6769\r\nIXI439,43.9179\r\nIXI440,48.1013\r\nIXI441,48.1013\r\nIXI443,69.1335\r\nIXI444,58.3765\r\nIXI445,60.0274\r\nIXI446,60.1095\r\nIXI447,72.8268\r\nIXI448,79.4141\r\nIXI449,46.8966\r\nIXI450,68.5092\r\nIXI451,41.1307\r\nIXI452,63.4031\r\nIXI453,63.4168\r\nIXI454,32.5777\r\nIXI455,68.4819\r\nIXI456,70.2478\r\nIXI458,70.7105\r\nIXI459,62.4257\r\nIXI460,60.8953\r\nIXI461,60.8953\r\nIXI465,69.2813\r\nIXI467,66.9487\r\nIXI468,67.6988\r\nIXI479,60.0164\r\nIXI480,60.0164\r\nIXI481,65.3306\r\nIXI482,72.0903\r\nIXI483,63.3922\r\nIXI484,66.3737\r\nIXI485,68.8515\r\nIXI486,63.8357\r\nIXI487,66.2752\r\nIXI488,69.0678\r\nIXI489,69.0678\r\nIXI490,70.141\r\nIXI491,75.9726\r\nIXI492,69.2895\r\nIXI493,67.7673\r\nIXI494,66.3326\r\nIXI495,60.8679\r\nIXI496,62.8036\r\nIXI497,72.3066\r\nIXI498,76.7803\r\nIXI499,82.1875\r\nIXI500,63.1814\r\nIXI501,67.1129\r\nIXI502,63.8768\r\nIXI503,65.0951\r\nIXI504,68.1506\r\nIXI505,62.245\r\nIXI506,69.6893\r\nIXI507,71.6632\r\nIXI508,60.9309\r\nIXI511,64.5777\r\nIXI512,64.7502\r\nIXI515,62.089\r\nIXI516,60.4107\r\nIXI518,75.8275\r\nIXI519,57.5058\r\nIXI521,70.8364\r\nIXI522,28.9336\r\nIXI523,63.4196\r\nIXI524,33.5168\r\nIXI525,41.5195\r\nIXI526,54.6092\r\nIXI528,40.5284\r\nIXI531,75.937\r\nIXI533,73.6044\r\nIXI534,69.6099\r\nIXI535,65.2594\r\nIXI536,58.8309\r\nIXI537,67.4552\r\nIXI538,81.9411\r\nIXI539,78.0726\r\nIXI544,58.0424\r\nIXI546,60.0739\r\nIXI549,25.0185\r\nIXI550,59.9069\r\nIXI551,70.5024\r\nIXI552,64.6626\r\nIXI554,70.1054\r\nIXI555,65.6674\r\nIXI556,55.6359\r\nIXI558,66.8583\r\nIXI559,67.8193\r\nIXI560,63.8686\r\nIXI562,42.7899\r\nIXI565,63.8166\r\nIXI566,42.9706\r\nIXI567,28.5613\r\nIXI568,33.6865\r\nIXI569,56.9637\r\nIXI572,41.9493\r\nIXI575,57.755\r\nIXI576,67.332\r\nIXI577,64.1862\r\nIXI578,57.8645\r\nIXI579,41.2156\r\nIXI582,29.7714\r\nIXI584,41.3279\r\nIXI585,28.1205\r\nIXI586,34.3655\r\nIXI587,33.5934\r\nIXI591,59.8932\r\nIXI592,48.0794\r\nIXI593,55.2197\r\nIXI594,61.9959\r\nIXI598,52.8871\r\nIXI599,39.384\r\nIXI600,38.9596\r\nIXI601,35.5154\r\nIXI603,66.0151\r\nIXI605,70.3326\r\nIXI606,60.5394\r\nIXI607,83.8111\r\nIXI608,52.4983\r\nIXI609,28.9966\r\nIXI610,56.6762\r\nIXI611,28.8706\r\nIXI612,33.9165\r\nIXI613,25.5852\r\nIXI614,26.9295\r\nIXI616,55.0883\r\nIXI617,39.1759\r\nIXI618,37.9493\r\nIXI619,24.8761\r\nIXI621,31.6167\r\nIXI622,59.6851\r\nIXI625,47.0691\r\nIXI626,64.2683\r\nIXI627,68.4846\r\nIXI629,59.2635\r\nIXI630,54.6858\r\nIXI631,41.3005\r\nIXI632,56.4381\r\nIXI633,36.7064\r\nIXI634,34.0561\r\nIXI635,25.5305\r\nIXI636,43.1567\r\nIXI639,86.1985\r\nIXI640,80.8022\r\nIXI641,68.1232\r\nIXI642,57.0678\r\nIXI644,56.8734\r\nIXI646,71.206\r\nIXI648,47.7235\r\nIXI651,50.3956\r\nIXI652,42.9897\r\nIXI653,46.2204\r\nIXI662,41.7413\r\n"
IXI002 = {"scan_id": "IXI002", "dataset": "IXI", "site": "Guys", "chronological_age": 35.8001, "sex": "F", "original_pred": 36.8402, "fname": "IXI002-Guys-0828-T1.nii.gz", "url": "https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10452/scans/T1/resources/NIfTI/files/IXI002-Guys-0828-T1.nii.gz", "md5": "b7873629cbb6eebfa8264179271da09c", "size": 6943524}
REF_JSON = "{\"antsxnet\":{\"brainAgeDeepBrainNet.h5\":\"50d76851567d502e38755892746473af\",\"brainExtractionRobustT1.h5\":\"515a171b701df6ab512735f02ea26b14\",\"croppedMni152.nii.gz\":\"2170efb430dcc82dc97860985c8f18df\",\"croppedMni152Priors.nii.gz\":\"5e8e851dc2f99cc9b4aac634b4590be0\",\"sixTissueOctantBrainSegmentationWithPriors1.h5\":\"61d5c2574913f3f2e4716fd0eb04a43a\",\"S_template3.nii.gz\":\"ba473fc2a89895be028ec798842d92b7\"},\"bang_weights\":{\"BrainAge_1.pth\":\"b19d69ebbf3dc46e27bd4215b890e03f\",\"BrainAge_2.pth\":\"8ed0e327a7b4d06c92c0884f08edfd0a\",\"BrainAge_3.pth\":\"9b4f95fdd0fdda5fecfc72fa8cdbeb38\",\"BrainAge_4.pth\":\"1abf4a1019d22b9912511a494c794336\",\"BrainAge_5.pth\":\"b0eee326c4494a3294809ef5b2749626\"},\"bang_code_md5\":\"fed48108580f032f95f8d1ed1505a461\",\"dbn_ref\":{\"IXI002\":{\"pred\":36.2744,\"gap_mean\":[0.00034,0.0,0.00328,0.00124,0.00589,0.0,0.0,0.0,0.0,0.00033,0.0,0.00083,0.0001,0.0,0.0046,0.00175,0.0,1e-05,0.07146,3e-05,0.00047,0.0,0.00982,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00158,0.0,0.0,0.00117,2e-05,0.02221,0.00072,0.00724,0.0,0.0,0.00112,0.00099,0.21199,0.0,0.0,0.0,0.0,1e-05,0.01098,0.0,0.0,0.00921,0.002,0.0,0.00029,0.10151,2e-05,0.0,0.0,0.0,0.0,0.0,0.00413,0.0,0.0,0.0,0.0,0.00282,0.0,0.00016,0.0,0.0,0.00078,0.0,0.00952,0.0027,0.12466,0.0,1e-05,0.0,0.0,0.0,0.0,0.00015,0.0,0.04128,0.00529,0.0,0.0,0.03921,7e-05,0.0,0.00811,4e-05,0.0,0.0,0.00022,1e-05,0.0,0.0,0.0,0.23805,5e-05,0.0,6e-05,0.0,1e-05,0.0,0.00352,0.0,0.0,0.0,0.0,0.92945,0.0,0.0,0.01167,6e-05,0.00012,0.07527,0.01048,0.0,0.0,0.11992,0.00013,0.0,9e-05,0.0,0.00039,0.0,0.0,0.00024,0.02465,0.00037,0.0,0.00231,8e-05,0.0,0.00021,0.0,9e-05,0.00011,0.0,0.00017,0.00183,0.0,0.00632,0.00026,0.0,0.0,0.00643,0.0,0.00099,0.0001,0.0,5.6917,0.001,0.01247,0.0,0.0,0.0,0.00026,0.0,0.2672,0.0011,0.00027,0.01026,0.0,0.0,0.0,0.0,0.00014,0.0,0.0,0.0,7e-05,0.0,0.00043,0.0,0.0021,0.0,0.00231,0.00022,0.0,0.0,7e-05,0.00413,0.0,0.0,0.00832,0.00476,0.0,0.0,0.0,0.00077,0.0,0.0,0.0,0.0029,0.00029,0.01039,0.0,0.00237,0.00175,0.00014,1e-05,0.0,0.0,0.00854,0.00013,0.00054,0.0,0.03746,0.00031,0.0,0.0,0.00098,0.0,0.00021,0.0,0.0,0.0,0.0,0.03152,0.00047,0.02171,0.00097,0.0,0.02747,0.00015,0.00158,0.01113,0.0001,2e-05,0.0,0.001,0.00406,0.00097,0.0,0.0,0.0,0.00066,0.0,0.0,0.0,0.0,0.0,0.00461,0.0001,0.0,0.0,0.00152,0.0,0.0158,0.04758,0.0,0.0,0.0,0.0,0.01699,0.00041,0.0,0.0,0.00301,0.00013,0.0,0.0,0.0,4e-05,0.10139,0.0,0.00119,0.00012,0.0,0.0,0.0023,0.16661,0.01059,0.00503,0.0,0.0,0.00314,5e-05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00611,0.00028,0.0,0.0,0.0,0.00023,0.0,0.00056,0.0,0.01305,0.00354,0.0,0.0,0.00131,0.00024,0.0,0.0,0.00066,0.0005,0.0,0.0,0.0,0.85363,0.0,0.0,2e-05,0.0,0.0,0.00058,0.00071,0.0,0.00822,0.0034,0.0,0.0,0.08796,0.00085,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00055,0.0,3e-05,0.0,0.0,0.01281,1e-05,0.00019,0.0,0.00206,0.0,0.0,0.00119,0.00058,0.0,0.24087,0.0,0.00058,3e-05,0.0,0.00164,0.0,0.00017,0.00204,0.0,0.0,0.0,0.0,0.0,0.00022,0.0,0.0,0.00071,4e-05,0.04649,0.00069,0.0,0.0,0.0,0.0,0.0,0.0,0.0043,0.06091,0.0,0.06174,0.0,0.00042,0.0,0.00081,0.00031,0.00039,0.0,0.14739,0.00011,0.00855,0.0,0.0003,0.0,0.0,0.00046,0.0,0.00033,0.09123,0.0,0.0,0.00089,0.0,0.00096,0.0,0.0,2e-05,0.00073,0.0,0.02193,0.0,0.0,6e-05,0.0,0.0008,0.02242,0.0,0.00834,0.00015,3e-05,0.00021,0.01311,0.00019,0.0,0.0,0.0,0.00013,7e-05,0.0,0.0,0.00103,0.0,0.00142,0.0,0.00396,0.00668,0.35809,0.00142,0.0,1.59413,0.00512,0.0,0.00384,0.0001,0.0,0.00063,0.0,0.00728,0.00041,0.0,0.00066,0.0002,0.00021,0.0,0.0012,0.00135,2e-05,0.29486,0.0008,0.0,1e-05,0.01375,0.0,0.13199,0.00261,3e-05,0.0,0.0001,0.00643,0.0,0.0,0.05177,0.00352,0.0,0.0,0.00067,0.0,0.0,0.00108,0.0,0.0,0.0,0.00017,6e-05,0.00576,0.0,0.0,0.0026,0.0,0.0,0.0,0.00023,0.0,0.00055,0.0,0.0,0.00029,0.0,0.00578,0.0,0.13897,0.0,0.0,0.0,1e-05,0.0,0.0,0.0,0.11,0.00032]},\"IXI012\":{\"pred\":46.2345,\"gap_mean\":[0.00134,0.0,0.0211,0.00474,0.0397,0.0,0.0,0.0,2e-05,0.00337,0.0,0.01028,3e-05,0.0,0.00339,0.01158,1e-05,2e-05,0.04827,0.00026,0.00392,0.0,0.00045,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.017,0.0,0.0,0.00013,0.0,0.00586,0.00042,0.00651,0.0,0.0,0.00085,0.00075,0.28359,0.0,0.0,0.0011,0.0,0.0,0.02952,0.00032,0.0,0.04437,0.0004,0.0,0.00025,0.15952,0.00036,0.0,0.0,0.0,0.00011,0.0,0.0103,0.0,0.0,0.0,0.0,0.0218,0.0,0.00045,0.00018,0.00064,3e-05,0.0,0.00132,0.00181,0.18508,0.0,2e-05,0.0,0.0,2e-05,0.0,2e-05,0.0,0.02563,0.03697,0.0,0.0,0.06122,0.00043,0.0,0.02859,2e-05,0.00018,0.0,0.0011,0.0,0.0,0.00044,0.0,0.32811,0.00093,0.0,0.0,0.00267,0.0,7e-05,0.00196,0.0,0.0,6e-05,0.00024,1.16034,0.0,0.0,0.00735,3e-05,0.00031,0.16281,0.0062,0.0,0.00057,0.17241,0.00786,1e-05,0.00038,3e-05,0.00045,0.0,0.0,0.00091,0.01183,0.00328,0.0,0.0006,0.0,0.00042,0.0018,0.0,0.0,0.0,0.0,0.00058,0.00042,0.0,0.00669,0.00029,0.0,0.0,0.00144,0.00022,0.00643,4e-05,0.00089,6.96401,0.00387,0.01092,0.0,0.0,0.0,0.00102,0.0,0.36523,0.0,0.0,0.03296,0.00014,0.0,0.0,0.0,0.00262,0.00076,0.0,0.0,0.0001,2e-05,0.00036,0.00089,0.00573,0.0,0.00677,0.00233,0.0,0.0001,0.00251,0.00367,0.0,0.0,0.00806,0.00302,0.0,0.0,0.0,0.00017,0.00023,0.0,0.0,0.00143,0.00088,0.01536,0.0,0.00198,0.00528,0.00012,0.0,0.00048,6e-05,0.01293,0.0,0.00102,0.0001,0.03182,9e-05,0.0,0.0,0.01259,0.0,0.00333,4e-05,0.00352,0.0,0.0,0.02074,0.00068,0.0727,0.00905,0.00043,0.02588,0.00091,0.00151,0.00924,4e-05,0.0,0.0,0.00513,0.01125,0.00115,0.00182,0.0,0.0,0.00026,6e-05,0.0,0.0,0.0,0.00048,0.00285,0.0,1e-05,0.0,0.00393,0.0,0.02033,0.10065,0.0,0.0,0.0,0.0,0.03384,0.00093,0.0,0.00018,0.01276,0.00252,0.0,1e-05,0.0,8e-05,0.17421,0.0,0.00243,0.00088,9e-05,0.0,0.00143,0.21939,0.02896,0.00709,0.00017,0.00032,0.00503,0.0023,0.0,0.0,0.00032,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00832,0.0,0.00021,0.0,0.0,0.00468,0.0,0.00087,0.00231,0.01039,0.02222,0.0,0.00514,0.00046,0.00017,0.0,0.00043,0.00303,0.00564,0.0,0.0,0.0,1.07336,0.0,0.0,7e-05,7e-05,0.0,0.00125,0.00017,0.0,0.00346,0.00361,0.0,0.0,0.1308,0.00047,0.00397,0.00011,0.0001,0.0,0.0,0.0,0.0,0.00249,0.0,0.0,0.0,0.00012,0.04509,8e-05,0.01043,0.0,0.00342,0.0,0.0,0.00374,0.0054,0.0,0.34044,0.0,0.00038,0.00182,0.00021,0.00189,0.0,0.00127,0.00623,0.0,0.0,0.0,0.00011,3e-05,0.00085,0.0,0.0,0.01134,0.00016,0.09269,0.00249,0.00528,0.0,0.00026,0.0,0.00023,0.0,0.00256,0.06498,0.00012,0.08693,0.0,0.00125,0.0,0.00696,0.00195,0.00122,0.0,0.24081,0.00428,0.00326,1e-05,0.00184,0.00122,0.0,0.0,5e-05,0.0016,0.07348,0.0,0.0,0.00018,0.0,0.00067,0.00012,0.0,0.0045,0.00789,0.0,0.02908,0.0,0.0,1e-05,0.0,0.00155,0.0241,0.0,0.04368,0.0,0.00169,0.00157,0.01026,0.00041,0.0,0.0,0.0,0.00167,0.00028,2e-05,0.0,0.00493,0.0,0.0053,0.0,0.00972,0.00953,0.39284,0.00751,0.0,1.98309,0.00956,0.00265,0.01491,0.0,1e-05,5e-05,0.0,0.0081,0.00141,0.0,0.00213,0.0,0.0004,0.0,0.00124,0.0001,0.0,0.169,0.00311,0.0,0.0,0.03671,0.0,0.13147,0.01714,1e-05,0.0,0.0,0.00667,0.0,0.0,0.0412,0.00943,0.0,0.0,0.0059,0.0,0.0,0.00041,0.0,0.0,6e-05,0.00721,0.00056,0.02652,0.0,0.0,0.01757,1e-05,0.0,0.0,0.00187,0.0,0.00122,0.0,0.0,0.00141,0.0,0.0156,0.0,0.20339,0.0,0.0,0.0,0.00178,0.0,2e-05,7e-05,0.15742,0.00968]},\"IXI013\":{\"pred\":49.2911,\"gap_mean\":[0.00287,0.0,0.03498,0.00962,0.02891,7e-05,0.00011,0.0,5e-05,8e-05,0.0,0.00223,0.00025,0.0,0.00824,0.01837,0.0,0.0,0.11683,0.00019,0.00342,8e-05,0.00155,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.01948,0.0,0.0,1e-05,0.0,0.03958,0.00024,0.0129,0.0,0.0,0.00603,0.00036,0.32952,0.00019,0.0,0.00157,0.0,0.0,0.03886,0.00012,0.0,0.0808,0.00149,0.0,0.00095,0.18738,0.00065,7e-05,0.0,0.0,0.0,0.0,0.01147,0.0,0.0,0.0,0.0,0.01888,0.0,0.00012,0.0,1e-05,0.00057,0.0,0.00154,0.00329,0.22475,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8e-05,0.01612,0.064,2e-05,0.0,0.07117,0.0,0.0,0.01968,0.00077,0.00039,0.0,0.00187,0.00011,0.0,0.00011,0.0,0.37583,0.0033,0.0,0.0,0.00331,1e-05,0.0,0.00098,0.0,0.0,9e-05,0.0,1.26055,1e-05,0.00037,0.00898,0.0,0.00349,0.20993,0.0168,0.0,0.00059,0.21318,0.0262,0.00013,0.00017,0.00075,0.00097,0.0,0.0,0.00206,0.00526,0.00526,0.0,0.00168,0.0,0.00033,0.00593,0.0,0.001,0.0,0.0,0.00394,0.00488,0.00022,0.00794,0.00374,0.0,0.0,0.00013,0.002,0.00516,0.0,0.00108,7.40337,0.00398,0.02736,0.0,0.0,0.0,0.00055,0.0,0.42409,0.00067,5e-05,0.04935,0.00026,0.0,0.0,0.0,0.014,0.0,0.00018,0.0001,0.0,0.0,0.00213,0.0,0.01618,0.0,0.00301,0.00731,0.0,0.00662,0.004,0.00358,0.0,0.0,0.03083,0.00446,0.0,0.00023,0.0,0.00118,0.0,0.0,0.0,0.00345,0.00092,0.01993,0.0,0.00565,0.00452,0.00014,0.0,0.0,0.00096,0.00732,0.0,0.00687,0.00017,0.06487,2e-05,5e-05,0.0,0.03994,0.0,0.00171,0.00018,0.00071,0.00025,1e-05,0.0678,0.00526,0.11318,0.02075,0.0,0.03452,0.00147,0.00427,0.03073,0.0,0.00011,0.0,0.01812,0.02701,0.00247,0.0,8e-05,0.0,0.00149,0.0,0.0,0.0,0.0,0.0,0.00095,0.00011,0.0,0.0011,0.00699,0.00052,0.02895,0.13276,0.0,0.0,0.0,0.0,0.05054,0.00357,0.00062,5e-05,0.01043,0.00235,0.0,0.0,0.0,0.0,0.22727,8e-05,0.00388,5e-05,0.0,0.0,0.00136,0.25071,0.02467,0.01733,0.00109,6e-05,0.02745,0.00414,0.0,0.0,7e-05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0215,1e-05,0.0,0.0,0.00026,0.00942,0.0,0.00558,0.00108,0.01132,0.04564,0.0,0.00037,0.00041,3e-05,0.0,0.00268,0.01435,0.02716,0.0,0.0,0.0,1.15905,0.00017,0.0,0.0059,0.00097,0.0,0.00753,0.00023,0.0,0.00878,0.00214,1e-05,0.0001,0.15973,0.00026,0.03455,0.0,0.0,0.0003,0.0,0.0,0.0,0.00239,0.0,0.0,7e-05,0.00111,0.06508,0.00023,0.02817,2e-05,0.00689,0.00013,0.0,0.00688,0.00176,0.0,0.39411,0.0,0.00325,0.00135,0.0,0.00358,0.0,0.00203,0.00563,0.0,0.0,0.0,0.00013,0.0,0.00422,0.0,0.00012,0.01839,0.00011,0.1174,0.00736,0.00258,0.0,0.0,0.0,0.00035,0.0,0.00011,0.07184,0.0,0.10388,0.0,0.00139,0.0,0.01213,0.00281,0.00182,0.00017,0.28447,0.00323,0.00332,0.00029,0.00015,2e-05,7e-05,0.0,0.0,0.00331,0.09681,2e-05,0.0,0.00047,3e-05,0.00044,0.00091,0.00564,0.00199,0.0005,0.0,0.03908,0.0,0.0,0.0,0.0,0.00109,0.0295,0.0,0.04376,0.0,0.03618,0.00493,0.02236,0.00342,0.0,0.0,0.0,0.00417,5e-05,2e-05,0.0,0.00307,0.0,0.00471,0.0,0.00629,0.01761,0.47712,0.01,0.0,2.13708,0.00881,0.00371,0.02376,0.0,6e-05,0.00028,0.00168,0.01095,0.00917,0.0,0.0021,0.01454,0.00125,0.0,0.00192,0.00487,0.0,0.16982,0.01108,0.00301,0.0,0.03184,0.0,0.13598,0.02462,0.00018,0.0,0.00038,0.03176,0.0,0.0,0.10259,0.02037,0.0,0.0,0.00491,0.0,0.0,0.0067,0.0,0.0,0.0,0.0105,0.00132,0.03898,0.0,0.0,0.02242,8e-05,0.0,0.0,0.0026,0.0,0.00161,0.0,0.00104,0.00152,0.0,0.02874,0.0,0.23809,0.0,0.0,0.0,0.0023,9e-05,0.0,0.00016,0.18081,0.03084]}},\"bang_ref\":{\"IXI002\":{\"as_is\":37.2,\"flip_LR\":35.5,\"flip_AP\":40.1,\"flip_SI\":42.7}}}"
ORIENT = "as_is"   # locked on IXI, never re-chosen here

In [ ]:
# ===== 3. Drive, logging, safety helpers =====
os.environ['TF_USE_LEGACY_KERAS'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import io, json, time, shutil, hashlib, pathlib, datetime
import numpy as np, pandas as pd, requests
from google.colab import drive, runtime
drive.mount('/content/drive')

DRIVE = pathlib.Path('/content/drive/MyDrive') / DRIVE_FOLDER
OUT = DRIVE / 'sald_outputs'
PREV = DRIVE / 'outputs'                      # IXI run: features used to train the heads
DBN_OUT, BANG_OUT, CACHE = OUT / 'dbn_embed', OUT / 'bang_embed', DRIVE / 'ANTsXNet'
WORK = pathlib.Path('/content/work')
for d in (DBN_OUT, BANG_OUT, CACHE, WORK / 't1', WORK / 'bang'):
    d.mkdir(parents=True, exist_ok=True)
LOG = OUT / 'run_log.txt'

manifest = pd.read_csv(io.StringIO(SALD_CSV))
REF = json.loads(REF_JSON)
N = len(manifest)

def log(msg):
    line = f'{datetime.datetime.now():%m-%d %H:%M:%S}  {msg}'
    print(line, flush=True)
    with open(LOG, 'a') as f:
        f.write(line + '\n')

def release():
    if AUTO_RELEASE_RUNTIME:
        log('releasing the Colab runtime (no more compute is used)')
        try:
            drive.flush_and_unmount()
        finally:
            runtime.unassign()

def stop(reason, ok=False):
    log(('PAUSED: ' if ok else 'STOPPED: ') + reason)
    if not ok:
        (OUT / 'STOPPED.txt').write_text(f'{datetime.datetime.now()}\n{reason}\n')
    release()
    raise Stop(reason)

def md5sum(p):
    h = hashlib.md5()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()

def s3_etag(p, part=8 * 1024 * 1024):
    # S3 multipart ETag: md5 of the concatenated per-part md5 digests, plus '-<parts>'
    digests = []
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(part), b''):
            digests.append(hashlib.md5(b).digest())
    return hashlib.md5(b''.join(digests)).hexdigest() + f'-{len(digests)}'

def fetch(url, dest, check, tries=5):
    # download to .part, verify with check(path) -> bool, then rename; retries with backoff
    dest = pathlib.Path(dest)
    if dest.exists() and check(dest):
        return dest
    last = None
    for i in range(tries):
        try:
            tmp = dest.with_name(dest.name + '.part')
            with requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, stream=True, timeout=(30, 300)) as r:
                r.raise_for_status()
                with open(tmp, 'wb') as f:
                    for b in r.iter_content(1 << 20):
                        f.write(b)
            if not check(tmp):
                raise IOError(f'checksum mismatch for {dest.name}')
            os.replace(tmp, dest)
            return dest
        except Exception as e:
            last = e
            time.sleep(10 * (i + 1))
    raise RuntimeError(f'download failed after {tries} tries: {url}  ({last!r})')

def put(src, dst):
    # copy to Drive under a temp name, then rename: a disconnect never leaves a half-written file
    tmp = dst.with_name(dst.name + '.part')
    shutil.copyfile(src, tmp)
    os.replace(tmp, dst)

def dbn_ok(p):
    try:
        d = np.load(p)
        return (d['gap'].shape == (80, 512) and d['dense'].shape == (80, 1024)
                and np.isfinite(d['gap']).all() and np.isfinite(d['dense']).all()
                and 0 < float(d['predicted_age']) < 120)
    except Exception:
        return False

def bang_ok(p):
    try:
        d = np.load(p)
        return d['feat'].shape == (5, 512) and np.isfinite(d['feat']).all() and np.isfinite(d['preds']).all()
    except Exception:
        return False

import torch
GPU = torch.cuda.is_available()
if (OUT / 'STOPPED.txt').exists():
    log('note: the previous session stopped with: ' + (OUT / 'STOPPED.txt').read_text().strip().replace('\n', ' | '))
    (OUT / 'STOPPED.txt').unlink()
free_gb = shutil.disk_usage('/content/drive/MyDrive').free / 1e9
log(f'SALD session start  runtime={"T4 GPU" if GPU else "CPU"}  drive_free={free_gb:.1f} GB  scans={N}')
if free_gb < 2:
    stop(f'Google Drive has only {free_gb:.1f} GB free; about 1.5 GB is needed for outputs.')

In [ ]:
# ===== 4. Load models + pre-flight reproduction checks (IXI002, same references as the IXI run) =====
from concurrent.futures import ThreadPoolExecutor
D1 = {s for s in manifest.scan_id if (DBN_OUT / f'{s}.npz').exists()}
D2 = {s for s in manifest.scan_id if (BANG_OUT / f'{s}.npz').exists()}
log(f'DeepBrainNet {len(D1)}/{N} done   BrainAgeNeXt {len(D2)}/{N} done')
NEED1, NEED2 = len(D1) < N, len(D2) < N
BANG_READY = GPU and NEED2

def t1_of(r):
    return fetch(r.url, WORK / 't1' / r.fname, lambda p: s3_etag(p) == r.etag)

import tensorflow as tf
for g in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(g, True)    # leave GPU memory for BrainAgeNeXt
import ants, tf_keras as keras
from antspynet.utilities import preprocess_brain_image, set_antsxnet_cache_directory, get_pretrained_network
set_antsxnet_cache_directory(str(CACHE))
h5 = get_pretrained_network('brainAgeDeepBrainNet')
if md5sum(h5) != REF['antsxnet']['brainAgeDeepBrainNet.h5']:
    stop('DeepBrainNet weights differ from the laptop copy.')
model = keras.models.load_model(h5)
feat_model = keras.Model(model.input, [model.get_layer('global_average_pooling2d_2').output,
                                       model.get_layer('dense_3').output, model.output])
SLICES = list(range(45, 125))

def dbn_scan(scan_id, t1_path, save=True):
    # returns (pred, gap-mean features, local prep path); with save=True also writes to Drive
    prep = preprocess_brain_image(ants.image_read(str(t1_path)), truncate_intensity=(0.01, 0.99),
                                  brain_extraction_modality='t1', template='croppedMni152',
                                  template_transform_type='antsRegistrationSyNQuickRepro[a]',
                                  do_bias_correction=True, do_denoising=True, verbose=False)
    img = prep['preprocessed_image'] * prep['brain_mask']
    img = (img - img.min()) / (img.max() - img.min())
    batch = np.zeros((len(SLICES), *img.shape[0:2], 3), dtype=np.float32)
    for j, idx in enumerate(SLICES):
        batch[j] = ants.slice_image(img, axis=2, idx=idx).numpy()[..., None]
    gap, dense, sp = feat_model.predict(batch, verbose=0)
    pred = float(np.median(sp[:, 0]))
    lp = WORK / f'{scan_id}_prep.nii.gz'
    ants.image_write(img, str(lp))
    if save:
        ln = WORK / f'{scan_id}.npz'
        np.savez_compressed(ln, gap=gap.astype(np.float16), dense=dense.astype(np.float16),
                            slice_pred=sp[:, 0].astype(np.float32), predicted_age=np.float32(pred))
        if not dbn_ok(ln):
            raise ValueError('saved features failed the shape/NaN check')
        put(lp, DBN_OUT / lp.name)
        put(ln, DBN_OUT / ln.name)          # npz last: it is what marks the scan as done
        ln.unlink()
        D1.add(scan_id)
    return pred, gap.astype(np.float32).mean(0), lp

log('pre-flight: re-running IXI002 through DeepBrainNet and comparing with the laptop ...')
ixi_t1 = fetch(IXI002['url'], WORK / 't1' / IXI002['fname'], lambda p: md5sum(p) == IXI002['md5'])
pred0, g0, prep0 = dbn_scan('IXI002', ixi_t1, save=False)
ref = REF['dbn_ref']['IXI002']
dp, c = abs(pred0 - ref['pred']), float(np.corrcoef(g0, np.array(ref['gap_mean']))[0, 1])
log(f'  reproduce IXI002: colab={pred0:.2f} laptop={ref["pred"]:.2f} (diff {dp:.2f} y)  feature r={c:.4f}')
if dp > 1.0 or c < 0.98:
    stop(f'pre-flight failed: DeepBrainNet does not reproduce the laptop (diff {dp:.2f} y, r={c:.3f}).')
for f in CACHE.iterdir():
    if f.name in REF['antsxnet'] and md5sum(f) != REF['antsxnet'][f.name]:
        stop(f'antspynet file {f.name} differs from the laptop copy.')
log('DeepBrainNet pre-flight passed')

VARIANTS = {'as_is': None, 'flip_LR': 2, 'flip_AP': 3, 'flip_SI': 4}   # tensor is (1, 1, X, Y, Z)
if BANG_READY:
    B = WORK / 'bang'
    fetch('https://raw.githubusercontent.com/FrancescoLR/BrainAgeNeXt/main/BrainAge_estimation.py',
          B / 'BrainAge_estimation.py', lambda p: md5sum(p) == REF['bang_code_md5'])
    for name, h in REF['bang_weights'].items():
        fetch(f'https://huggingface.co/FrancescoLR/BrainAgeNeXt/resolve/main/{name}', B / name,
              lambda p, h=h: md5sum(p) == h)
    sys.path.insert(0, str(B))
    from BrainAge_estimation import MedNeXtEncReg, prepare_transforms   # official file, MD5-verified
    bang = []
    for i in range(1, 6):
        m = MedNeXtEncReg().cuda()
        m.load_state_dict(torch.load(B / f'BrainAge_{i}.pth', map_location='cuda'))
        bang.append(m.eval())
    tfm = prepare_transforms()

    def bang_run(prep_path, flip_dim):
        x = torch.as_tensor(np.asarray(tfm({'image': str(prep_path)})['image']))[None].cuda()
        if flip_dim is not None:
            x = torch.flip(x, dims=[flip_dim])
        feats, preds = [], []
        with torch.no_grad():
            for m in bang:
                f = torch.flatten(m.global_avg_pool(m.mednextv1(x)), 1)   # same path as forward()
                feats.append(f[0].float().cpu().numpy())
                preds.append(float(m.regression_fc(f).squeeze()))
        return np.stack(feats), np.array(preds, dtype=np.float32)

    log('pre-flight: BrainAgeNeXt on IXI002, all 4 orientations, vs the laptop ...')
    for k, refv in REF['bang_ref']['IXI002'].items():
        got = float(np.median(bang_run(prep0, VARIANTS[k])[1]))
        log(f'  reproduce BrainAgeNeXt IXI002 {k}: colab={got:.1f} laptop={refv:.1f}')
        if abs(got - refv) > 1.5:
            stop(f'BrainAgeNeXt does not reproduce the laptop ({k}: {got:.1f} vs {refv:.1f}).')
    log(f'BrainAgeNeXt pre-flight passed; orientation locked from the IXI run: {ORIENT}')
prep0.unlink(missing_ok=True)

In [ ]:
# ===== 5. Main loop: each SALD scan -> DeepBrainNet -> BrainAgeNeXt =====
class Guard:
    def __init__(self, name):
        self.name, self.fail, self.row = name, 0, 0
    def ok(self):
        self.row = 0
    def failed(self, sid, e):
        self.fail += 1
        self.row += 1
        log(f'  {sid} {self.name} FAILED: {e!r}')
        if self.row >= 3:
            stop(f'3 scans in a row failed in {self.name} (last: {sid}: {e!r}).')
        if self.fail > max(5, 0.05 * N):
            stop(f'{self.fail} scans failed in {self.name} (>5%).')

g1, g2 = Guard('DeepBrainNet'), Guard('BrainAgeNeXt')
p1s, e1s, e2s = [], [], []

def watch(preds, errs, name, max_mae):
    # no laptop reference exists for SALD: catch broken preprocessing instead of real-but-large errors
    if len(errs) >= 20 and np.mean(errs[-20:]) > max_mae:
        stop(f'{name} MAE over the last 20 SALD scans is {np.mean(errs[-20:]):.1f} y (> {max_mae}) - '
             'more likely broken preprocessing than a real effect; check before continuing.')
    if len(preds) >= 20 and np.std(preds[-20:]) < 2:
        stop(f'{name} predictions have collapsed (SD {np.std(preds[-20:]):.2f} y over the last 20 scans).')

def do_bang(r):
    feats, preds = bang_run(DBN_OUT / f'{r.scan_id}_prep.nii.gz', VARIANTS[ORIENT])
    ln = WORK / f'{r.scan_id}_bang.npz'
    np.savez_compressed(ln, feat=feats.astype(np.float16), preds=preds,
                        predicted_age=np.float32(np.median(preds)), orientation=ORIENT)
    if not bang_ok(ln):
        raise ValueError('saved features failed the shape/NaN check')
    put(ln, BANG_OUT / f'{r.scan_id}.npz')
    ln.unlink()
    D2.add(r.scan_id)
    return float(np.median(preds))

b2s = []
todo = [r for r in manifest.itertuples() if r.scan_id not in D1 or (BANG_READY and r.scan_id not in D2)]
log(f'{len(todo)} scans to process this session')
pool, futs, t0 = ThreadPoolExecutor(max_workers=3), {}, time.time()
for i, r in enumerate(todo):
    for nxt in todo[i:i + 4]:                 # download the next scans while this one runs
        if nxt.scan_id not in D1 and nxt.scan_id not in futs:
            futs[nxt.scan_id] = pool.submit(t1_of, nxt)
    p1 = p2 = None
    if r.scan_id not in D1:
        try:
            t1 = futs.pop(r.scan_id).result()
            p1, _, lp = dbn_scan(r.scan_id, t1)
            lp.unlink(missing_ok=True)
            t1.unlink(missing_ok=True)
            g1.ok()
            p1s.append(p1)
            e1s.append(abs(p1 - r.chronological_age))
            watch(p1s, e1s, 'DeepBrainNet', 20)
        except Stop:
            raise
        except Exception as e:
            g1.failed(r.scan_id, e)
            continue
    if BANG_READY and r.scan_id in D1 and r.scan_id not in D2:
        try:
            p2 = do_bang(r)
            g2.ok()
            b2s.append(p2)
            e2s.append(abs(p2 - r.chronological_age))
            watch(b2s, e2s, 'BrainAgeNeXt', 15)
        except Stop:
            raise
        except Exception as e:
            g2.failed(r.scan_id, e)
    if (i + 1) % 5 == 0:
        rate = (time.time() - t0) / (i + 1)
        log(f'working: DeepBrainNet {len(D1)}/{N}  BrainAgeNeXt {len(D2)}/{N}  '
            f'(last {r.scan_id}: age {r.chronological_age:.0f}'
            + (f', DBN {p1:.1f}' if p1 is not None else '') + (f', BANG {p2:.1f}' if p2 is not None else '')
            + f')  {rate:.0f} s/scan, ~{(len(todo) - i - 1) * rate / 3600:.1f} h left'
            + (f'  running MAE DBN {np.mean(e1s):.2f} / BANG {np.mean(e2s):.2f}' if e1s and e2s else ''))
pool.shutdown(wait=False)

log(f'finished this pass: DeepBrainNet {len(D1)}/{N}  BrainAgeNeXt {len(D2)}/{N}')
if N - len(D1) > 0.05 * N:
    stop(f'{N - len(D1)} scans still missing DeepBrainNet features; Run all again to retry them.')
if not GPU:
    stop('DeepBrainNet is complete. Switch to Runtime > Change runtime type > T4 GPU, then Run all.', ok=True)
if N - len(D2) > 0.05 * N:
    stop(f'{N - len(D2)} scans still missing BrainAgeNeXt features; Run all again to retry them.')

## Stage 3 - SALD results
Raw MAE of both CNNs on SALD, plus ridge heads trained **only on IXI Guys+HH** (features from the
earlier IXI run on Drive) and applied to SALD unchanged.

In [ ]:
# ===== 6. Stage 3 - evaluate on SALD; IXI-trained heads applied unchanged =====
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def feats(dbn_path, bang_path):
    a, b = np.load(dbn_path), np.load(bang_path)
    return dict(dbn_raw=float(a['predicted_age']), bang_raw=float(b['predicted_age']),
                bang_sd=float(np.std(b['preds'])), gap=a['gap'].astype(np.float32).mean(0),
                dense=a['dense'].astype(np.float32).mean(0), bangf=b['feat'].astype(np.float32).mean(0),
                **{f'bang_model{i + 1}': float(v) for i, v in enumerate(b['preds'])})

recs = []
for r in manifest.itertuples():
    a, b = DBN_OUT / f'{r.scan_id}.npz', BANG_OUT / f'{r.scan_id}.npz'
    if a.exists() and b.exists() and dbn_ok(a) and bang_ok(b):
        recs.append(dict(scan_id=r.scan_id, age=r.chronological_age, sex=r.sex, **feats(a, b)))
sald = pd.DataFrame(recs)
log(f'stage 3: {len(sald)} SALD scans with both feature sets')

mae = lambda p, y: float(np.mean(np.abs(np.asarray(p) - np.asarray(y))))
def row(name, p):
    p, y = np.asarray(p), sald['age'].to_numpy()
    return {'model': name, 'MAE': mae(p, y), 'mean_error': float(np.mean(p - y)),
            'r': float(np.corrcoef(p, y)[0, 1]), 'n': len(y)}
table = [row('DeepBrainNet raw', sald['dbn_raw']), row('BrainAgeNeXt raw', sald['bang_raw'])]

train = pd.read_csv(io.StringIO(IXI_TRAIN_CSV))
tr = []
for r in train.itertuples():
    a, b = PREV / 'dbn_embed' / f'{r.scan_id}.npz', PREV / 'bang_embed' / f'{r.scan_id}.npz'
    if a.exists() and b.exists():
        tr.append(dict(age=r.chronological_age, **feats(a, b)))
tr = pd.DataFrame(tr)
log(f'IXI training features found on Drive: {len(tr)}/{len(train)}')
if len(tr) >= 450:
    X = lambda df, cols: np.hstack([np.vstack(df[c].to_numpy()) for c in cols])
    for name, cols in {'DBN dense1024 + ridge': ['dense'], 'BANG 512 + ridge': ['bangf'],
                       'DBN gap + BANG + ridge': ['gap', 'bangf']}.items():
        head = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-1, 4, 20)))
        head.fit(X(tr, cols), tr['age'])
        sald[f'pred {name}'] = head.predict(X(sald, cols))
        table.append(row(name + ' (trained on IXI)', sald[f'pred {name}']))
else:
    log('not enough IXI features on Drive (outputs/ of the IXI run) - heads skipped, raw results only')

res = pd.DataFrame(table).set_index('model').round(3)
res.to_csv(OUT / 'sald_mae.csv')
sald['age_bin'] = pd.cut(sald['age'], [19, 30, 40, 50, 60, 70, 81], right=False).astype(str)
by_bin = sald.groupby('age_bin').apply(lambda g: pd.Series({
    'n': len(g), 'DBN': mae(g['dbn_raw'], g['age']), 'BANG': mae(g['bang_raw'], g['age'])})).round(2)
sald.drop(columns=['gap', 'dense', 'bangf']).to_csv(OUT / 'sald_predictions.csv', index=False)
log('SALD MAE (held out for BrainAgeNeXt; heads never saw SALD):\n' + res.to_string())
log('MAE by age bin:\n' + by_bin.to_string())
(OUT / 'DONE.txt').write_text(f'{datetime.datetime.now()}\n' + res.to_string() + '\n')
log('ALL DONE. Run the export cell below, then send the zip.')
release()

## Export for the GitHub repo
Run **only this cell** after the run has finished (a CPU runtime is enough). It downloads
`iaudit_sald_outputs.zip`.

In [ ]:
# ===== 7. Export SALD results for the repo (run ONLY this cell; CPU runtime is enough) =====
import pathlib, zipfile
import pandas as pd
from google.colab import drive, files
drive.mount('/content/drive')
OUT = pathlib.Path('/content/drive/MyDrive/iAuditAge/sald_outputs')
NEED = ['sald_predictions.csv', 'sald_mae.csv', 'run_log.txt']
missing = [f for f in NEED if not (OUT / f).exists()]
if missing:
    raise SystemExit(f'not found in {OUT}: {missing} - has the run finished?')
print(pd.read_csv(OUT / 'sald_mae.csv').to_string(index=False))
bang = sorted((OUT / 'bang_embed').glob('*.npz'))
zpath = pathlib.Path('/content/iaudit_sald_outputs.zip')
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as zf:
    for name in NEED + ['DONE.txt']:
        if (OUT / name).exists():
            zf.write(OUT / name, name)
    for f in bang:
        zf.write(f, f'bang_embed/{f.name}')
print(f'{zpath.name}: {zpath.stat().st_size / 1e6:.1f} MB ({len(bang)} BrainAgeNeXt feature files)')
files.download(str(zpath))